# Data quality

The pipeline checks its own output. Fourteen checks (one SQL file each, in `sql/dq/`) run as the **last step of every pipeline run**.
A check returns the number of rows that break its rule: **0 means pass**. If any check fails, the run ends with the
names of the failing checks and the batch is reported as failed.

| Kind | Checks |
|---|---|
| Silver, for the batch | the batch has patients and encounters; no duplicate patient, encounter or condition ids; no visit ends before it starts |
| Gold | no duplicate patients or encounters; no visit ends before it starts; every visit points to a patient, a facility, a doctor and a date that exist; every silver encounter of the batch reached the fact table |

This notebook shows the results for every batch, then **proves that the checks can fail**: it plants a bad row in the gold fact table,
shows which checks catch it and that the pipeline stops, and removes the row again.

## 1. Setup

In [1]:
import json
import sys
from pathlib import Path

import boto3

# Works whether Jupyter starts in the project root or in notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "infra"))

# Shared helpers (see infra/): deploy.py deploys the workflow, sql_files.py combines the check files
import deploy
import sql_files
from aws_helpers import (SETTINGS, execution_steps, run_athena, start_execution, wait_for_execution, whoami,
                         account_id)

CONFIG = deploy.load_config()
SESSION = boto3.Session(region_name=CONFIG["region"])
CHECKS = sql_files.read_checks(ROOT / "sql" / "dq")
INGEST_DATES = [f"2026-11-{n:02d}" for n in range(1, SETTINGS["number_of_batches"] + 1)]

print("identity:", whoami())
print(f"{len(CHECKS)} checks:", ", ".join(CHECKS))

identity: user/Ahmed
14 checks: gold_batch_missing_in_fact, gold_dim_patient_duplicate_keys, gold_fact_duplicate_keys, gold_fact_end_before_start, gold_fact_without_date, gold_fact_without_facility, gold_fact_without_patient, gold_fact_without_practitioner, silver_condition_duplicate_keys, silver_encounter_duplicate_keys, silver_encounter_end_before_start, silver_encounter_has_rows, silver_patient_duplicate_keys, silver_patient_has_rows


## 2. Deploy the workflow that contains the checks
`deploy_state_machine` inserts the one combined query into the workflow. It adds three steps after the merges:
`DataQualityChecks` (run the query), `ReadDataQualityResult` (read its answer) and `EvaluateDataQuality`
(`OK` leads to success, anything else to a failure that names the failing checks).

In [2]:
STATE_MACHINE_ARN = deploy.deploy_state_machine(SESSION, CONFIG)
definition = deploy.state_machine_definition(CONFIG, account_id())
check = boto3.client("stepfunctions").validate_state_machine_definition(definition=json.dumps(definition), type="STANDARD")
assert check["result"] == "OK", check["diagnostics"]
print("OK  workflow deployed and valid; steps:", " -> ".join(definition["States"]))

updated state machine pf-batch-pipeline


OK  workflow deployed and valid; steps: Ingest -> BronzeToSilver -> PrepareDateParameter -> MergeDimPatient -> MergeDimFacility -> MergeDimPractitioner -> MergeFactEncounter -> DataQualityChecks -> ReadDataQualityResult -> EvaluateDataQuality -> DataQualityFailed -> PipelineSucceeded -> PipelineFailed


## 3. Results for every batch
One row per check, one column per batch. The same query the workflow uses, but returning every check instead of only the names of failures.

In [3]:
DETAILED = sql_files.detailed_checks_query(list(CHECKS.values()))


def run_checks(ingest_date):
    result = run_athena(DETAILED, parameters=[sql_files.quote_date(ingest_date)], timeout=300)
    return {name: int(failures) for name, failures in result["rows"]}


results = {ingest_date: run_checks(ingest_date) for ingest_date in INGEST_DATES}

print(f"{'check':<38}" + "".join(f"{d[5:]:>8}" for d in INGEST_DATES))
for name in CHECKS:
    print(f"{name:<38}" + "".join(f"{results[d][name]:>8}" for d in INGEST_DATES))

assert all(failures == 0 for per_batch in results.values() for failures in per_batch.values())
print("\nOK  all", len(CHECKS), "checks pass for all", len(INGEST_DATES), "batches")

check                                    11-01   11-02   11-03   11-04
gold_batch_missing_in_fact                   0       0       0       0
gold_dim_patient_duplicate_keys              0       0       0       0
gold_fact_duplicate_keys                     0       0       0       0
gold_fact_end_before_start                   0       0       0       0
gold_fact_without_date                       0       0       0       0
gold_fact_without_facility                   0       0       0       0
gold_fact_without_patient                    0       0       0       0
gold_fact_without_practitioner               0       0       0       0
silver_condition_duplicate_keys              0       0       0       0
silver_encounter_duplicate_keys              0       0       0       0
silver_encounter_end_before_start            0       0       0       0
silver_encounter_has_rows                    0       0       0       0
silver_patient_duplicate_keys                0       0       0       0
silver

## 4. Can the checks fail? Plant a bad row
One row is added to `fact_encounter` with several deliberate mistakes: it ends **before** it starts, and it points to a patient,
a facility and a doctor that do not exist. The checks must name exactly these problems.
The `try ... finally` guarantees the row is deleted afterwards.

In [4]:
BAD_ID = "dq-test-bad-row"
INSERT_BAD_ROW = f'''
INSERT INTO pf_gold.fact_encounter VALUES (
  '{BAD_ID}', 'no-such-patient', 'no-such-facility', 'no-such-npi', 'AMB',
  TIMESTAMP '2020-01-02 10:00:00', TIMESTAMP '2020-01-02 09:00:00', DATE '2020-01-02', DATE '2020-01-02',
  -1.0, -0.04, false, false, NULL, NULL, NULL
)'''
DELETE_BAD_ROW = f"DELETE FROM pf_gold.fact_encounter WHERE encounter_id = '{BAD_ID}'"

EXPECTED_FAILURES = {"gold_fact_end_before_start", "gold_fact_without_patient",
                     "gold_fact_without_facility", "gold_fact_without_practitioner"}

run_athena(DELETE_BAD_ROW)                       # start clean in case an earlier run was interrupted
try:
    run_athena(INSERT_BAD_ROW)
    with_bad_row = run_checks("2026-11-01")
    failing = {name for name, failures in with_bad_row.items() if failures}
    print("failing checks:", sorted(failing))
    assert failing == EXPECTED_FAILURES, failing
    print("OK  the checks caught exactly the planted mistakes")
finally:
    run_athena(DELETE_BAD_ROW)

assert not any(run_checks("2026-11-01").values())
print("OK  bad row removed; all checks pass again")

failing checks: ['gold_fact_end_before_start', 'gold_fact_without_facility', 'gold_fact_without_patient', 'gold_fact_without_practitioner']
OK  the checks caught exactly the planted mistakes


OK  bad row removed; all checks pass again


## 5. The pipeline must stop on bad data
With the bad row planted again, the **whole workflow** runs for batch 1 (about 9 minutes). It must end as `FAILED` at the
quality step and say which checks failed. Then the row is removed.

In [5]:
try:
    run_athena(INSERT_BAD_ROW)
    execution = wait_for_execution(start_execution(STATE_MACHINE_ARN, {"batch": 1}))
    steps = execution_steps(execution["executionArn"])
finally:
    run_athena(DELETE_BAD_ROW)

print("status:", execution["status"])
print("error :", execution.get("error"))
print("cause :", execution.get("cause"))
print("steps :", " -> ".join(steps))

assert execution["status"] == "FAILED" and execution["error"] == "DataQualityFailed"
for name in EXPECTED_FAILURES:
    assert name in execution["cause"], name
assert steps[-2:] == ["DataQualityFailed", "PipelineFailed"] and "MergeFactEncounter" in steps
print("OK  the pipeline ran all the loads, then stopped at the quality step and named the failing checks")

assert not any(run_checks("2026-11-01").values())
print("OK  the bad row is gone; all checks pass")

status: FAILED
error : DataQualityFailed
cause : These data-quality checks failed: gold_fact_end_before_start, gold_fact_without_facility, gold_fact_without_patient, gold_fact_without_practitioner
steps : Ingest -> BronzeToSilver -> PrepareDateParameter -> MergeDimPatient -> MergeDimFacility -> MergeDimPractitioner -> MergeFactEncounter -> DataQualityChecks -> ReadDataQualityResult -> EvaluateDataQuality -> DataQualityFailed -> PipelineFailed
OK  the pipeline ran all the loads, then stopped at the quality step and named the failing checks


OK  the bad row is gone; all checks pass


## What this is for

| Where | How |
|---|---|
| Every pipeline run | the three quality steps after the merges: a failure ends the run as `FAILED` with `DataQualityFailed` and the check names |
| Adding a check | add a SQL file to `sql/dq/` that returns `check_name` and `failures` (see the existing files). The next deployment puts it into the workflow; the tests make sure the file has the right shape |
| Reading the results | this notebook (the table in section 3), or a failed execution's *cause* in the Step Functions console |

Limits worth knowing: the checks run **after** the gold tables are loaded, so a failing batch has already been written. The run is marked failed so that
nobody trusts it, and fixing the problem and releasing the batch again overwrites it (every step is repeatable).